# Local inference with fine-tuned adapter

This notebook demonstrates that the fine-tuned Phi-4-mini adapter can run **completely offline** on a local device.

> **Note:** This notebook loads the model locally. Ensure you have ~8 GB free RAM/VRAM.

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
repo_root = Path.cwd().parents[0]
load_dotenv(repo_root / '.env', override=True)

In [ ]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from azure.identity import DefaultAzureCredential
from azure.storage.blob import BlobServiceClient
from azure_infra import download_model
from iss_utils import create_classification_prompt, parse_classification_response

In [ ]:
# Configuration from environment variables
STORAGE_ACCOUNT = os.getenv("FINETUNE_STORAGE_ACCOUNT")
CONTAINER_NAME = "finetune"  # 3-63 chars required; "ft" was too short and failed silently
BASE_MODEL_ID = "microsoft/Phi-4-mini-instruct"
ADAPTER_LOCAL_PATH = "models/ft"

print(f"Storage Account: {STORAGE_ACCOUNT}")
print(f"Base Model:      {BASE_MODEL_ID}")

## Local Demo: Offline Inference

Demonstrate that the fine-tuned model works completely offline on a single report.

In [ ]:
# Demo: Run on a single report LOCALLY (completely offline)
# This shows the fine-tuned model can run on-device without cloud

# Download adapter from Blob Storage if not present locally
if not os.path.exists(f"{ADAPTER_LOCAL_PATH}/ft/adapter"):
    print("Downloading fine-tuned adapter from Blob Storage...")
    download_model(STORAGE_ACCOUNT, CONTAINER_NAME, ADAPTER_LOCAL_PATH)
    print("Download complete.")

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading model on local device: {device}")

# Load tokenizer and base model, then apply LoRA adapter (~30s on Mac M1/M2)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float16 if device != "cpu" else torch.float32,
    trust_remote_code=True
).to(device)

# Load LoRA adapter
ft_model = PeftModel.from_pretrained(base_model, f"{ADAPTER_LOCAL_PATH}/ft/adapter")
ft_model.eval()
print("\nFine-tuned model loaded! Running demo classification...")

# Run a single ISS report through the fine-tuned model
from iss_utils import get_evaluation_dataset, fetch_report
eval_dataset = get_evaluation_dataset()
sample_item = eval_dataset[0]
sample_report = fetch_report(sample_item["date"])

if sample_report and sample_report.get("report_text"):
    prompts = create_classification_prompt(sample_report["report_text"])
    messages = [
        {"role": "system", "content": prompts["system"]},
        {"role": "user", "content": prompts["user"]}
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,  # returns {input_ids, attention_mask} - silences pad/eos warning
    ).to(device)
    with torch.no_grad():
        outputs = ft_model.generate(**inputs, max_new_tokens=300, do_sample=False)
    input_len = inputs["input_ids"].shape[1]
    response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

    print(f"\nDate: {sample_item['date']}")
    print(f"Expected Severity: {sample_item['expected_severity']}")
    print(f"\nModel Response:")
    print("-" * 60)
    print(response[:800])
else:
    print("Could not fetch sample report for demo.")

## Summary

**Results:**

We achieved a **+5% accuracy improvement** (45.7% → 51.4%) through knowledge distillation-a first gain that demonstrates the pattern works. The fine-tuned Phi-4-mini now outperforms its base version on ISS incident classification.

**What we demonstrated:**

1. **Serverless GPU Training** - Used Azure Container Apps with A100 GPUs for fine-tuning, avoiding the need for dedicated GPU infrastructure
2. **Knowledge Distillation** - Transferred domain expertise from DeepSeek-V3.2 (teacher) to Phi-4-mini (student)
3. **Synthetic Data Generation** - Augmented limited real data with 500+ generated examples
4. **Edge Deployment** - The fine-tuned model runs completely offline on local devices

**The Key Pattern:**

Your edge device may only run a quantized 4-bit model, but you still want the best possible weights. This workflow offloads the compute-intensive training to serverless cloud GPUs, then deploys the optimized model locally. You get the best of both worlds: cloud-scale training with edge-scale inference.

**Further Improvements to Try:**

1. **Balance training data** - The model over-predicts "caution". Ensure equal representation of all severity levels in synthetic data generation.
2. **Increase training steps** - Try 200-300 steps (2-3 epochs) instead of 100 to give the model more exposure to each class.
3. **Add contrastive examples** - Include pairs of similar reports with different severities to teach decision boundaries.

**Next Steps:**

To deploy this model on resource-constrained devices (phones, embedded systems), use **Microsoft Olive** to quantize the fine-tuned adapter to INT4/INT8. This reduces memory footprint by 4-8x while preserving most of the accuracy gains from fine-tuning.